In [1]:
# Import required libraries
import sys
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime

# Import the LLM debiasing analyzer
from LLM_debias import LLMPositionBiasAnalyzer

print("📚 Libraries imported successfully!")
print(f"📅 Experiment started at: {datetime.now()}")


📚 Libraries imported successfully!
📅 Experiment started at: 2025-07-26 16:07:16.705561


In [2]:
import pandas as pd

# Define file paths
base_path = 'data/ml-1m/'
ratings_file = base_path + 'ratings.dat'
users_file = base_path + 'users.dat'
movies_file = base_path + 'movies.dat'

ratings = pd.read_csv(
    ratings_file,
    sep='::',
    engine='python',
    names=['UserID', 'MovieID', 'Rating', 'Timestamp'],
    encoding='latin-1'
)

users = pd.read_csv(
    users_file,
    sep='::',
    engine='python',
    names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'],
    encoding='latin-1'
)

movies = pd.read_csv(
    movies_file,
    sep='::',
    engine='python',
    names=['MovieID', 'Title', 'Genres'],
    encoding='latin-1'
)
# Merge ratings with users
ratings_users = pd.merge(ratings, users, on='UserID')

# Merge with movies
full_data = pd.merge(ratings_users, movies, on='MovieID')

print("\nCombined Dataset:\n", full_data[full_data['UserID'] == 1][['Title', 'Timestamp']])


Combined Dataset:
                                                 Title  Timestamp
0              One Flew Over the Cuckoo's Nest (1975)  978300760
1                    James and the Giant Peach (1996)  978302109
2                                 My Fair Lady (1964)  978301968
3                              Erin Brockovich (2000)  978300275
4                                Bug's Life, A (1998)  978824291
5                          Princess Bride, The (1987)  978302268
6                                      Ben-Hur (1959)  978302039
7                           Christmas Story, A (1983)  978300719
8              Snow White and the Seven Dwarfs (1937)  978302268
9                            Wizard of Oz, The (1939)  978301368
10                        Beauty and the Beast (1991)  978824268
11                                        Gigi (1958)  978301752
12                      Miracle on 34th Street (1947)  978302281
13                    Ferris Bueller's Day Off (1986)  978302124
14   

In [3]:
# Initialize the analyzer
print("🔧 INITIALIZING LLM BIAS ANALYZER")
print("=" * 50)

analyzer = LLMPositionBiasAnalyzer(
    data=full_data,
    data_name="movie_lens",
    model="gpt-3.5-turbo",
    backend="openai",
    list_size = 20,
    num_bias_users = 5,
    api_tier="tier_2"  # Adjust based on your OpenAI tier
)

print("✅ Analyzer initialized successfully!")
print(f"📊 Dataset: {analyzer.data_name}")
print(f"🤖 Model: {analyzer.model}")
print(f"🔌 Backend: {analyzer.backend}")
print(f"📈 API Tier: {analyzer.api_tier}")

# Check data loading
print(f"\n📋 Data loaded: {len(analyzer.data)} interactions")
print(f"👥 Unique users: {analyzer.data['UserID'].nunique()}")
print(f"🎬 Unique items: {analyzer.data['MovieID'].nunique()}")


🔧 INITIALIZING LLM BIAS ANALYZER
📊 User filtering results:
  Total users in dataset: 6040
  Users with ≥6 items: 6040
  Filtered out: 0 users
✅ Selected 5 bias users and 200 evaluation users
   All selected users have ≥6 items for reliable evaluation
Initialized LLM Bias Analyzer:
  Model: gpt-3.5-turbo
  Backend: openai
  API Tier: tier_2
  Rate Limits: 5000 RPM, 2000000 TPM
  Max Workers: 25
  Batch Size: 50
  Request Delay: 0.030s
✅ Analyzer initialized successfully!
📊 Dataset: movie_lens
🤖 Model: gpt-3.5-turbo
🔌 Backend: openai
📈 API Tier: tier_2

📋 Data loaded: 1000209 interactions
👥 Unique users: 6040
🎬 Unique items: 3706


In [4]:
# bias_analysis =  analyzer.compute_bias_analysis(5,None,True,3,20)
# print(bias_analysis)

In [5]:
prebias_gpt35_movielens = {'avg_primacy': 0.33199999999999996,
 'avg_recency': 0.044,
 'avg_middle': 1.6239999999999999}

prebias_gpt4_movielens = {'avg_primacy': 0.6440000000000001,
 'avg_recency': 0.348,
 'avg_middle': 1.008}

prebias_gpt4o_movielens = {'avg_primacy': 0.58,
 'avg_recency': 0.35200000000000004,
 'avg_middle': 1.068}

In [6]:
# Step 1: Configure experiment parameters and run the complete evaluation
print("\n🔧 COMPLETE DEBIASING EXPERIMENT")
print("=" * 50)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 15     # Number of randomization trials per user
batch_size = 20        # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Generate checkpoint filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_file = f"experiment_checkpoint_{timestamp}.json"

print(f"💾 Checkpoint file: {checkpoint_file}")

# Run the complete evaluation pipeline
# This will automatically:
# 1. Split users into bias detection and evaluation sets
# 2. Run bias detection on bias detection users
# 3. Calculate propensity scores from detected bias
# 4. Evaluate on evaluation users using the calculated propensity scores
print("\n🚀 RUNNING COMPLETE EVALUATION PIPELINE...")
print("This will:")
print("1. 📊 Select separate users for bias detection vs evaluation")
print("2. 🔍 Run bias detection on bias detection users")  
print("3. ⚖️ Calculate propensity scores from detected bias")
print("4. 📈 Evaluate on evaluation users using calculated propensity scores")
print("5. 💾 Save all raw data for future reanalysis")

results = analyzer.evaluate_our_method_batched(
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    use_parallel=True,
    precalculated_bias= prebias_gpt35_movielens,
    checkpoint_file = "evaluation_checkpoint_movielens_trails15_4.json"
)

print("\n✅ EVALUATION COMPLETED!")



🔧 COMPLETE DEBIASING EXPERIMENT
🎯 Candidates per evaluation: 20
🔄 Trials per user: 15
📦 Batch size: 20
💾 Checkpoint file: experiment_checkpoint_20250726_160720.json

🚀 RUNNING COMPLETE EVALUATION PIPELINE...
This will:
1. 📊 Select separate users for bias detection vs evaluation
2. 🔍 Run bias detection on bias detection users
3. ⚖️ Calculate propensity scores from detected bias
4. 📈 Evaluate on evaluation users using calculated propensity scores
5. 💾 Save all raw data for future reanalysis
📁 Checkpoint file: evaluation_checkpoint_movielens_trails15_4.json
📂 Resuming from checkpoint: 0 users already completed
API Tier: tier_2 (RPM: 5000, TPM: 2000000)
Max workers - Bias: 25, Trials: 12, Users: 3
📊 Using bias analysis from checkpoint
👥 Total users: 200, Completed: 0, Remaining: 200

🔄 Processing batch 1/8 (25 users)
Evaluating 25 users in parallel with max_workers=3...

Running 15 randomization trials (with raw data preservation)...
Executing 15 trials in parallel with max_workers=12...



Trials (batch 1/1):   0%|                                | 0/15 [00:00<?, ?it/s]


Executing 15 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50




Trials (batch 1/1):   0%|                                | 0/15 [00:00<?, ?it/s]

Trials (batch 1/1):  27%|██████▍                 | 4/15 [00:02<00:05,  2.14it/s]

Trials (batch 1/1):  13%|███▏                    | 2/15 [00:02<00:14,  1.12s/it]

Trials (batch 1/1):  40%|█████████▌              | 6/15 [00:03<00:03,  2.37it/s]

Trials (batch 1/1):  47%|███████████▏            | 7/15 [00:03<00:02,  2.78it/s]

Trials (batch 1/1):  53%|████████████▊           | 8/15 [00:03<00:02,  2.75it/s]

Trials (batch 1/1):  53%|████████████▊           | 8/15 [00:03<00:03,  2.14it/s]

Trials (batch 1/1):  60%|██████████████▍         | 9/15 [00:04<00:02,  2.55it/s]

Trials (batch 1/1):  67%|███████████████▎       | 10/15 [00:04<00:01,  3.11it/s]

Trials (batch 1/1):  80%|██████████████████▍    | 12/15 [00:04<00:00,  3.80it/s]

Trials (batch 1/1):  73%|████████████████▊      | 11/15 [00:04<00:01,  2.94it/s]

Trials (batch 1/1):  80%|██████████████████▍    | 12/15 [00:05<00:01,  2.39it/s]

Trials (batch 

Completed 15 successful trials out of 15 attempted

Running 15 randomization trials (with raw data preservation)...
Executing 15 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50


...

Completed 15 successful trials out of 15 attempted


Trials (batch 1/1):  73%|████████████████▊      | 11/15 [00:01<00:00,  7.88it/s]

Trials (batch 1/1):   7%|█▌                      | 1/15 [00:00<00:13,  1.03it/s]

Trials (batch 1/1):  20%|████▊                   | 3/15 [00:01<00:03,  3.33it/s]

Trials (batch 1/1):  33%|████████                | 5/15 [00:01<00:01,  5.36it/s]

Trials (batch 1/1):  87%|███████████████████▉   | 13/15 [00:02<00:00,  5.46it/s]

Trials (batch 1/1):  93%|█████████████████████▍ | 14/15 [00:02<00:00,  5.00it/s]

Trials (batch 1/1): 100%|███████████████████████| 15/15 [00:04<00:00,  3.56it/s]


Completed 15 successful trials out of 15 attempted




Trials (batch 1/1): 100%|███████████████████████| 15/15 [00:03<00:00,  4.31it/s]


Completed 15 successful trials out of 15 attempted
Progress: 25/25 users completed
✅ Completed evaluation of 25 users
✅ Batch 8 completed. Progress: 200/200 users

📊 Computing final metrics from 200 user results...

OUR METHOD EVALUATION RESULTS vs BENCHMARKS

Our Method Results:
  Accuracy:    0.1900 ± 0.3923
  NDCG@1:      0.1900 ± 0.3923
  NDCG@5:      0.3866 ± 0.3816
  NDCG@10:     0.4471 ± 0.3384
  NDCG@20:     0.5075 ± 0.2686
  Number of evaluations: 200

Benchmark Results (Accuracy) - From Paper:
Method          Movie Dataset  
------------------------------
Raw Output      0.2740±0.0593
Bootstrapping   0.2537
STELLA          0.2976
Our Method      0.1900±0.3923

Accuracy Comparison (Movie Dataset):
----------------------------------------
Our Method vs Raw Output:    -0.0840
Our Method vs Bootstrapping: -0.0637
Our Method vs STELLA:        -0.1076

NDCG Analysis:
----------------------------------------
NDCG@1 = Accuracy: 0.1900
NDCG@5:  0.3866 (203.5% of NDCG@1)
NDCG@10: 0.447

In [14]:
# Reprocess checkpoint data with custom propensity formula
print("\n🔄 REPROCESSING CHECKPOINT DATA WITH CUSTOM PROPENSITY")
print("=" * 60)

# Define custom propensity function: exp(2*(b1+b2+b3))
def custom_propensity_function(position, N, bias_analysis):
    """Custom propensity function: exp(2*(b1+b2+b3))"""
    import math
    
    # Get bias values from the saved bias analysis
    avg_primacy = bias_analysis['bias_scores']['avg_primacy']
    avg_recency = bias_analysis['bias_scores']['avg_recency'] 
    avg_middle = bias_analysis['bias_scores']['avg_middle']
    
    # Calculate normalized bias values
    expected_primacy = 0.025 * N
    expected_recency = 0.025 * N  
    expected_middle = 0.05 * N
    
    B_prim = (avg_primacy - expected_primacy) / expected_primacy
    B_rec = (avg_recency - expected_recency) / expected_recency
    B_mid = (avg_middle - expected_middle) / expected_middle
    
    # Normalized position: x = 0 (first) ... 1 (last)
    x = position / (N - 1)
    
    # Calculate bias terms
    primacy_term = B_prim * (1 - x)  # b1
    recency_term = B_rec * x         # b2
    middle_term = B_mid * (1 - 4 * (x - 0.5) ** 2)  # b3
    
    # Custom formula: exp(2*(b1+b2+b3))
    propensity_value = math.exp((primacy_term + recency_term + middle_term)/10)
    
    # Return inverse propensity weight
    return 1.0 / propensity_value

# Load the checkpoint file with saved raw data
checkpoint_file = "evaluation_checkpoint_bias20_movielens.json"
print(f"📁 Loading checkpoint: {checkpoint_file}")

# Check if checkpoint exists and analyze it
if os.path.exists(checkpoint_file):
    checkpoint_status = analyzer.analyze_checkpoint_file(checkpoint_file)
    print(f"\n📊 Checkpoint contains {checkpoint_status['total_results']} user results")
    
    # Create custom propensity scores using the saved bias analysis
    # First load the checkpoint to get bias analysis
    checkpoint_data = analyzer._load_checkpoint(checkpoint_file)
    bias_analysis = checkpoint_data.get('bias_analysis', {})
    
    if bias_analysis:
        print(f"\n⚖️ Creating custom propensity scores with formula: exp(2*(b1+b2+b3))")
        
        # Create custom propensity scores for 20 candidates (matching your evaluation)
        N = 20
        custom_propensity_scores = {}
        
        for pos in range(N):
            weight = custom_propensity_function(pos, N, bias_analysis)
            custom_propensity_scores[pos] = weight
        
        print(f"📊 Created custom propensity scores for {len(custom_propensity_scores)} positions")
        print("🔍 Sample custom weights:")
        for i in range(min(5, len(custom_propensity_scores))):
            print(f"  Position {i}: {custom_propensity_scores[i]:.4f}")
        
        # Reapply debiasing with custom propensity scores
        print(f"\n🚀 Reprocessing {checkpoint_status['total_results']} users with custom propensity...")
        
        reprocessed_results = analyzer.reapply_debiasing_from_checkpoint(
            checkpoint_file=checkpoint_file,
            new_propensity_scores=custom_propensity_scores,
            aggregation_method="mean",
            save_results_to=f"reprocessed_custom_propensity_{timestamp}.json"
        )
        
        if 'recomputed_evaluation' in reprocessed_results:
            print("\n🎉 REPROCESSING COMPLETED!")
            print("=" * 40)
            
            eval_results = reprocessed_results['recomputed_evaluation']
            print(f"📈 CUSTOM PROPENSITY RESULTS:")
            print(f"  Accuracy:  {eval_results['accuracy']['mean']:.4f} ± {eval_results['accuracy']['std']:.4f}")
            print(f"  NDCG@1:    {eval_results['ndcg_1']['mean']:.4f} ± {eval_results['ndcg_1']['std']:.4f}")
            print(f"  NDCG@5:    {eval_results['ndcg_5']['mean']:.4f} ± {eval_results['ndcg_5']['std']:.4f}")
            print(f"  NDCG@10:   {eval_results['ndcg_10']['mean']:.4f} ± {eval_results['ndcg_10']['std']:.4f}")
            print(f"  NDCG@20:   {eval_results['ndcg_20']['mean']:.4f} ± {eval_results['ndcg_20']['std']:.4f}")
            
            # Compare with original bias analysis if available
            original_bias = reprocessed_results['original_bias_analysis']['bias_scores']
            print(f"\n🧠 Original bias used:")
            print(f"  Primacy: {original_bias['avg_primacy']:.3f}")
            print(f"  Recency: {original_bias['avg_recency']:.3f}")
            print(f"  Middle:  {original_bias['avg_middle']:.3f}")
            
            print(f"\n💾 Results saved to: reprocessed_custom_propensity_{timestamp}.json")
        else:
            print("❌ Error in reprocessing")
            
    else:
        print("❌ No bias analysis found in checkpoint")
else:
    print(f"❌ Checkpoint file not found: {checkpoint_file}")
    print("💡 Make sure you've run the evaluation first to generate the checkpoint file")



🔄 REPROCESSING CHECKPOINT DATA WITH CUSTOM PROPENSITY
📁 Loading checkpoint: evaluation_checkpoint_bias20_movielens.json
📊 CHECKPOINT FILE ANALYSIS: evaluation_checkpoint_bias20_movielens.json
👥 Users completed: 200
📈 Total results: 200
🧠 Has bias analysis: Yes

📊 PERFORMANCE DISTRIBUTION:
Accuracy:  μ=0.3600, σ=0.4800
NDCG@1:    μ=0.3600, σ=0.4800
NDCG@5:    μ=0.5377, σ=0.4071
NDCG@10:   μ=0.5811, σ=0.3622
NDCG@20:   μ=0.6215, σ=0.3046

🏆 PERFORMANCE INSIGHTS:
Best accuracy:  1.0000 (User 1028)
Worst accuracy: 0.0000 (User 512)

📈 ACCURACY DISTRIBUTION:
High (≥0.8): 72/200 (36.0%)
Med (0.4-0.8): 0/200 (0.0%)
Low (<0.4): 128/200 (64.0%)

🧠 BIAS ANALYSIS:
Primacy: 0.33199999999999996
Recency: 0.044
Middle:  1.6239999999999999
Propensity scores: 20 positions

💾 FILE SIZE: 52.8 MB
📁 File: evaluation_checkpoint_bias20_movielens.json

📊 Checkpoint contains 200 user results

⚖️ Creating custom propensity scores with formula: exp(2*(b1+b2+b3))
📊 Created custom propensity scores for 20 positio

Recomputing: 100%|██████████████████████████| 200/200 [00:00<00:00, 1731.05it/s]

✅ Successfully recomputed 200 users

📊 RECOMPUTED RESULTS:
Accuracy:  0.3600 ± 0.4800
NDCG@1:    0.3600 ± 0.4800
NDCG@5:    0.5394 ± 0.4055
NDCG@10:   0.5812 ± 0.3621
NDCG@20:   0.6216 ± 0.3045
💾 Recomputed results saved to: reprocessed_custom_propensity_20250706_130856.json

🎉 REPROCESSING COMPLETED!
📈 CUSTOM PROPENSITY RESULTS:
  Accuracy:  0.3600 ± 0.4800
  NDCG@1:    0.3600 ± 0.4800
  NDCG@5:    0.5394 ± 0.4055
  NDCG@10:   0.5812 ± 0.3621
  NDCG@20:   0.6216 ± 0.3045

🧠 Original bias used:
  Primacy: 0.332
  Recency: 0.044
  Middle:  1.624

💾 Results saved to: reprocessed_custom_propensity_20250706_130856.json


In [6]:
# Step 2: Create default propensity scores based on detected bias
print("\n⚖️ CREATING DEFAULT PROPENSITY SCORES")
print("=" * 50)

# Extract bias values
primacy_bias = bias_results['primacy_bias']
middle_bias = bias_results['middle_bias']
recency_bias = bias_results['recency_bias']

# Create propensity scores using inverse propensity weighting
propensity_scores = analyzer.create_propensity_scores_dict(
    N=num_candidates,
    primacy_bias=primacy_bias,
    middle_bias=middle_bias,
    recency_bias=recency_bias
)

print(f"📊 Created propensity scores for {len(propensity_scores)} positions")
print("\n🔍 Sample propensity scores:")
for i in range(min(10, len(propensity_scores))):
    print(f"  Position {i}: {propensity_scores[i]:.4f}")

# Show bias mitigation weights
print("\n⚖️ BIAS MITIGATION WEIGHTS:")
print(f"Primacy positions (0-{int(0.1 * num_candidates)-1}): {propensity_scores[0]:.4f}")
print(f"Middle positions: {propensity_scores[num_candidates//2]:.4f}")
print(f"Recency positions ({int(0.9 * num_candidates)}-{num_candidates-1}): {propensity_scores[num_candidates-1]:.4f}")



⚖️ CREATING DEFAULT PROPENSITY SCORES


NameError: name 'bias_results' is not defined

In [ ]:
# Step 3: Run evaluation with raw data preservation
print("\n🚀 RUNNING EVALUATION WITH RAW DATA PRESERVATION")
print("=" * 50)

# Evaluation configuration
num_users = 50  # Start with 50 users for testing
num_trials = 20  # Number of randomization trials per user
batch_size = 10  # Process users in batches

print(f"👥 Evaluating {num_users} users")
print(f"🎯 {num_candidates} candidates per user")
print(f"🔄 {num_trials} trials per user")
print(f"📦 Batch size: {batch_size}")

# Generate checkpoint filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_file = f"experiment_checkpoint_{timestamp}.json"

print(f"💾 Checkpoint file: {checkpoint_file}")

# Run batched evaluation
results = analyzer.evaluate_our_method_batched(
    num_users=num_users,
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    precalculated_bias=bias_results,  # Use our calculated bias
    batch_size=batch_size,
    checkpoint_file="evaluation_checkpoint_bias20_movielens.json",
    use_parallel=True,
    max_workers_users=3,
    max_workers_trials=5
)

print("\n✅ EVALUATION COMPLETED!")


In [ ]:
# Step 4: Analyze results
print("\n📊 ANALYZING RESULTS")
print("=" * 50)

if results and 'evaluation_results' in results:
    eval_results = results['evaluation_results']
    
    print("🎯 PERFORMANCE METRICS:")
    print(f"Accuracy:  {eval_results['accuracy']['mean']:.4f} ± {eval_results['accuracy']['std']:.4f}")
    print(f"NDCG@1:    {eval_results['ndcg_1']['mean']:.4f} ± {eval_results['ndcg_1']['std']:.4f}")
    print(f"NDCG@5:    {eval_results['ndcg_5']['mean']:.4f} ± {eval_results['ndcg_5']['std']:.4f}")
    print(f"NDCG@10:   {eval_results['ndcg_10']['mean']:.4f} ± {eval_results['ndcg_10']['std']:.4f}")
    print(f"NDCG@20:   {eval_results['ndcg_20']['mean']:.4f} ± {eval_results['ndcg_20']['std']:.4f}")
    
    print(f"\n📈 EVALUATION SUMMARY:")
    print(f"Users evaluated: {eval_results['accuracy']['num_evaluations']}")
    print(f"Total trials: {eval_results['accuracy']['num_evaluations'] * num_trials}")
    
    # Bias analysis
    if 'bias_analysis' in results:
        bias_analysis = results['bias_analysis']
        print(f"\n🔍 BIAS ANALYSIS:")
        print(f"Primacy bias: {bias_analysis['primacy_bias']:.4f}")
        print(f"Middle bias:  {bias_analysis['middle_bias']:.4f}")
        print(f"Recency bias: {bias_analysis['recency_bias']:.4f}")
        
        # Calculate bias strength
        max_bias = max(bias_analysis['primacy_bias'], bias_analysis['recency_bias'])
        bias_strength = max_bias / bias_analysis['middle_bias'] if bias_analysis['middle_bias'] > 0 else float('inf')
        print(f"Bias strength: {bias_strength:.2f}x")
        
        if bias_strength > 1.5:
            print("⚠️  Strong positional bias detected - debiasing is important!")
        elif bias_strength > 1.2:
            print("⚡ Moderate positional bias detected - debiasing recommended")
        else:
            print("✅ Minimal positional bias detected")
    
else:
    print("❌ No results available - check for errors in evaluation")


In [ ]:
# Step 5: Save and analyze checkpoint data
print("\n💾 CHECKPOINT DATA ANALYSIS")
print("=" * 50)

# Check if checkpoint file exists
if os.path.exists(checkpoint_file):
    checkpoint_analysis = analyzer.analyze_checkpoint_file(checkpoint_file)
    
    print("📋 CHECKPOINT SUMMARY:")
    print(f"File size: {checkpoint_analysis['file_size']}")
    print(f"Users with data: {checkpoint_analysis['users_with_data']}")
    print(f"Total raw trials: {checkpoint_analysis['total_raw_trials']}")
    print(f"Average trials per user: {checkpoint_analysis['avg_trials_per_user']:.1f}")
    
    print("\n🔍 RAW DATA PRESERVATION:")
    print(f"Raw LLM outputs preserved: {checkpoint_analysis['total_raw_trials']} trials")
    print(f"Reanalysis ready: ✅")
    
    # Performance insights
    if 'performance_insights' in checkpoint_analysis:
        insights = checkpoint_analysis['performance_insights']
        print("\n📊 PERFORMANCE INSIGHTS:")
        print(f"Best user accuracy: {insights['best_user_accuracy']:.4f}")
        print(f"Worst user accuracy: {insights['worst_user_accuracy']:.4f}")
        print(f"Accuracy variance: {insights['accuracy_variance']:.4f}")
        
        if insights['accuracy_variance'] > 0.1:
            print("📈 High variance - users have diverse recommendation patterns")
        else:
            print("📉 Low variance - consistent performance across users")
    
    print(f"\n💡 TIP: Use '{checkpoint_file}' for reapplying different debiasing formulas!")
    
else:
    print(f"❌ Checkpoint file '{checkpoint_file}' not found")
